# 💱 Dólar Paralelo Bolivia — Modelado ARIMA y Sistema de Alertas

**Contexto:**  
Bolivia mantiene un tipo de cambio oficial fijo de Bs 6.96/USD desde 2011. Sin embargo, desde 2022 la escasez de dólares generó un mercado paralelo con prima creciente. Este sistema monitorea ese mercado y emite alertas cuando el TC se desvía de su trayectoria esperada.

**En uso real en Kapitalya** — los datos publicados aquí son sintéticos generados con `src/anonimizar_datos.py`.

**Pipeline:**
1. Carga y análisis exploratorio de la serie
2. Test de estacionariedad (ADF + KPSS)
3. Identificación de órdenes p, d, q con ACF/PACF
4. Selección automática por AIC
5. Diagnóstico de residuos
6. Forecast a 14 días con intervalos de confianza
7. Lógica de alertas sobre el forecast

In [ ]:
# ─── DEPENDENCIAS ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

import itertools
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
SEED = 42

# Colores tema Bolivia
COLOR_TC    = '#E74C3C'   # Rojo
COLOR_PRED  = '#2E86C1'   # Azul
COLOR_ALERT = '#F39C12'   # Naranja

print('✅ Librerías cargadas')

## 1. Carga de Datos

In [ ]:
# ─── CARGA ───────────────────────────────────────────────────────────────────
# Los datos reales fueron anonimizados con src/anonimizar_datos.py
# Este notebook usa los datos sintéticos generados por ese script
#
# Para generar datos sintéticos:
#   python src/anonimizar_datos.py --demo
#
# Para anonimizar tus propios datos reales:
#   python src/anonimizar_datos.py --input mis_datos.csv --output datos/anonimizados.csv

df = pd.read_csv('datos/datos_sinteticos.csv', parse_dates=['fecha'])
df = df.sort_values('fecha').reset_index(drop=True)

# La columna puede llamarse 'tc_paralelo' o 'indice_tc_paralelo' según anonimización
col_tc = 'tc_paralelo' if 'tc_paralelo' in df.columns else 'indice_tc_paralelo'
serie  = df.set_index('fecha')[col_tc].asfreq('B')  # Frecuencia días hábiles
serie  = serie.interpolate(method='time')            # Rellenar feriados

print(f'Serie: {len(serie)} observaciones')
print(f'Período: {serie.index[0].date()} → {serie.index[-1].date()}')
print(f'\nEstadísticos:')
print(f'  Media:  {serie.mean():.4f}')
print(f'  Mín:    {serie.min():.4f}  ({serie.idxmin().date()})')
print(f'  Máx:    {serie.max():.4f}  ({serie.idxmax().date()})')
print(f'  Σ (std): {serie.std():.4f}')

# Calcular la prima sobre el TC oficial
TC_OFICIAL = 6.96
if serie.mean() > 5:  # Solo si los datos no están normalizados
    prima_pct = (serie.iloc[-1] / TC_OFICIAL - 1) * 100
    print(f'\n  TC oficial: {TC_OFICIAL}')
    print(f'  TC paralelo actual: {serie.iloc[-1]:.4f}')
    print(f'  Prima: {prima_pct:.1f}%')

In [ ]:
# ─── GRÁFICA EXPLORATORIA ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Serie completa
ax = axes[0]
ax.fill_between(serie.index, serie.values, serie.min() * 0.98,
                 alpha=0.12, color=COLOR_TC)
ax.plot(serie.index, serie.values, color=COLOR_TC, lw=2)

# Media móvil 30 días
ma30 = serie.rolling(30).mean()
ax.plot(ma30.index, ma30.values, color='#2E86C1', lw=1.5,
         linestyle='--', label='Media móvil 30d', alpha=0.8)

# Bandas ±1σ rolling
sigma_roll = serie.rolling(30).std()
ax.fill_between(serie.index,
                 ma30 - sigma_roll,
                 ma30 + sigma_roll,
                 alpha=0.08, color='#2E86C1', label='Banda ±1σ (30d)')

ax.set_title('Tipo de Cambio Paralelo Bolivia (datos sintéticos)\n'
              'Datos reales anonimizados — uso en Kapitalya', fontweight='bold')
ax.set_ylabel('TC Paralelo (BOB/USD)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Retornos diarios
retornos = serie.pct_change().dropna() * 100
ax2 = axes[1]
colores_ret = [COLOR_TC if r < 0 else '#27AE60' for r in retornos.values]
ax2.bar(retornos.index, retornos.values, color=colores_ret, alpha=0.7, width=1)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(retornos.mean() + 2*retornos.std(), color=COLOR_ALERT,
             linestyle='--', lw=1.5, label='±2σ (umbral alerta)')
ax2.axhline(retornos.mean() - 2*retornos.std(), color=COLOR_ALERT,
             linestyle='--', lw=1.5)
ax2.set_title('Variación Diaria del TC Paralelo (%)', fontweight='bold')
ax2.set_ylabel('Cambio diario (%)')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('img/01_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nVariación diaria:')
print(f'  Media:    {retornos.mean():.4f}%')
print(f'  Std:      {retornos.std():.4f}%')
print(f'  Días con variación > 2σ: {(retornos.abs() > 2*retornos.std()).sum()}')

## 2. Test de Estacionariedad

In [ ]:
# ─── ADF + KPSS ──────────────────────────────────────────────────────────────
def test_estacionariedad(serie, nombre):
    print(f'\n{"─"*55}')
    print(f'  {nombre}')
    print(f'{"─"*55}')

    # ADF: H0 = tiene raíz unitaria (no estacionaria)
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(serie.dropna(), autolag='AIC')
    print(f'  ADF  | Estadístico: {adf_stat:.4f} | p-value: {adf_p:.4f}', end=' ')
    if adf_p < 0.05:
        print('→ Rechazamos H0 → ESTACIONARIA ✅')
    else:
        print('→ No rechazamos H0 → NO ESTACIONARIA ⚠️')

    # KPSS: H0 = es estacionaria
    kpss_stat, kpss_p, _, kpss_crit = kpss(serie.dropna(), regression='c', nlags='auto')
    print(f'  KPSS | Estadístico: {kpss_stat:.4f} | p-value: {kpss_p:.4f}', end=' ')
    if kpss_p > 0.05:
        print('→ No rechazamos H0 → ESTACIONARIA ✅')
    else:
        print('→ Rechazamos H0 → NO ESTACIONARIA ⚠️')

    if adf_p < 0.05 and kpss_p > 0.05:
        conclusion = 'ESTACIONARIA — usar d=0'
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        conclusion = 'NO ESTACIONARIA — aplicar d=1'
    else:
        conclusion = 'Señal ambigua — preferir d=1 por precaución'
    print(f'  → Conclusión: {conclusion}')
    return adf_p, kpss_p

adf_p, kpss_p = test_estacionariedad(serie, 'Serie original')
d_optimo = 0 if (adf_p < 0.05 and kpss_p > 0.05) else 1

if d_optimo == 1:
    test_estacionariedad(serie.diff().dropna(), 'Primera diferencia (d=1)')

print(f'\n📌 Orden de integración seleccionado: d = {d_optimo}')

## 3. Identificación de Órdenes p, q

In [ ]:
# ─── ACF / PACF ──────────────────────────────────────────────────────────────
serie_estac = serie.diff(d_optimo).dropna() if d_optimo > 0 else serie

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(serie_estac, lags=30, ax=axes[0],
          title='ACF — Función de Autocorrelación\n→ Identifica orden q (MA component)')
plot_pacf(serie_estac, lags=30, ax=axes[1], method='ywm',
           title='PACF — Autocorrelación Parcial\n→ Identifica orden p (AR component)')

for ax in axes:
    ax.set_xlabel('Lag (días)')

plt.tight_layout()
plt.savefig('img/02_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Guía de lectura:')
print('  ACF: si decae lentamente → componente AR dominante')
print('  ACF: si corta en lag k → q = k (MA)')
print('  PACF: si corta en lag k → p = k (AR)')
print('  Ambas con decaimiento lento → ARIMA con d=1')

## 4. Selección Automática del Mejor ARIMA

In [ ]:
# ─── GRID SEARCH POR AIC ─────────────────────────────────────────────────────
def seleccionar_arima(serie, max_p=4, max_d=2, max_q=4):
    """Prueba todas las combinaciones ARIMA(p,d,q) y devuelve la de menor AIC."""
    resultados = []
    combinaciones = list(itertools.product(
        range(max_p + 1), range(max_d + 1), range(max_q + 1)
    ))
    print(f'Evaluando {len(combinaciones)} combinaciones ARIMA(p,d,q)...')

    for p, d, q in combinaciones:
        if p == 0 and q == 0:
            continue
        try:
            modelo = SARIMAX(serie, order=(p, d, q),
                             enforce_stationarity=False,
                             enforce_invertibility=False)
            ajuste = modelo.fit(disp=False)
            resultados.append({
                'orden': (p, d, q),
                'aic':   round(ajuste.aic, 2),
                'bic':   round(ajuste.bic, 2),
            })
        except Exception:
            pass

    df_res = pd.DataFrame(resultados).sort_values('aic').reset_index(drop=True)
    print('\nTop 5 por AIC:')
    print(df_res.head().to_string(index=False))
    return df_res.iloc[0]['orden'], df_res

mejor_orden, tabla_aic = seleccionar_arima(serie.dropna())
print(f'\n🏆 Mejor modelo: ARIMA{mejor_orden}')

In [ ]:
# ─── AJUSTAR MODELO FINAL ─────────────────────────────────────────────────────
modelo_final = SARIMAX(
    serie.dropna(),
    order=mejor_orden,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(modelo_final.summary().tables[0])
print(modelo_final.summary().tables[1])

## 5. Diagnóstico de Residuos

In [ ]:
# ─── DIAGNÓSTICO ──────────────────────────────────────────────────────────────
residuos = modelo_final.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Residuos en el tiempo
axes[0, 0].plot(residuos.index, residuos.values, color='#2E86C1', lw=1)
axes[0, 0].axhline(0, color='red', linestyle='--', lw=1)
axes[0, 0].axhline(2*residuos.std(),  color=COLOR_ALERT, linestyle=':', lw=1, label='±2σ')
axes[0, 0].axhline(-2*residuos.std(), color=COLOR_ALERT, linestyle=':', lw=1)
axes[0, 0].set_title('Residuos en el tiempo', fontweight='bold')
axes[0, 0].legend(fontsize=9)

# 2. Histograma
res_norm = (residuos - residuos.mean()) / residuos.std()
axes[0, 1].hist(res_norm, bins=30, density=True, alpha=0.7,
                 color='#2E86C1', label='Residuos')
x = np.linspace(-4, 4, 100)
axes[0, 1].plot(x, stats.norm.pdf(x), 'r-', lw=2, label='Normal teórica')
_, p_norm = stats.normaltest(residuos.dropna())
axes[0, 1].set_title(f'Distribución residuos\nTest normalidad p={p_norm:.3f}',
                      fontweight='bold')
axes[0, 1].legend(fontsize=9)

# 3. QQ Plot
stats.probplot(residuos.dropna(), dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot', fontweight='bold')

# 4. ACF residuos
plot_acf(residuos.dropna(), lags=20, ax=axes[1, 1],
          title='ACF Residuos\n(deben estar dentro de las bandas)')

plt.tight_layout()
plt.savefig('img/03_diagnostico_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

# Test Ljung-Box
lb = acorr_ljungbox(residuos.dropna(), lags=[10], return_df=True)
lb_p = lb['lb_pvalue'].values[0]
print(f'\nTest Ljung-Box (autocorrelación en residuos):')
print(f'  p-value: {lb_p:.4f}', end=' ')
print('→ Sin autocorrelación ✅' if lb_p > 0.05 else '→ Hay autocorrelación residual ⚠️')
print(f'  Normalidad residuos p-value: {p_norm:.4f}', end=' ')
print('→ Normales ✅' if p_norm > 0.05 else '→ No normales (fat tails) ⚠️')

## 6. Forecast a 14 Días y Sistema de Alertas

In [ ]:
# ─── FORECAST ─────────────────────────────────────────────────────────────────
HORIZONTE = 14  # días hábiles (~2 semanas)

forecast    = modelo_final.get_forecast(steps=HORIZONTE)
fc_mean     = forecast.predicted_mean
fc_ci80     = forecast.conf_int(alpha=0.20)   # IC 80%
fc_ci95     = forecast.conf_int(alpha=0.05)   # IC 95%

# Generar fechas de forecast (días hábiles)
fechas_fc = pd.bdate_range(
    start=serie.dropna().index[-1] + pd.Timedelta(days=1),
    periods=HORIZONTE
)
fc_mean.index = fechas_fc
fc_ci80.index = fechas_fc
fc_ci95.index = fechas_fc

# ─── LÓGICA DE ALERTAS ───────────────────────────────────────────────────────
# Alerta ALTA:    TC actual > banda superior IC 80%
# Alerta MEDIA:   TC actual > media proyectada + 0.5σ
# Alerta BAJA:    Volatilidad 7d > 2 × volatilidad histórica
# Normal:         dentro de bandas

tc_actual       = serie.dropna().iloc[-1]
banda_superior  = fc_ci80.iloc[0, 1]   # Upper IC 80% para el día siguiente
banda_inferior  = fc_ci80.iloc[0, 0]   # Lower IC 80%
vol_7d          = serie.dropna().tail(7).pct_change().std()
vol_historica   = serie.dropna().pct_change().std()

if tc_actual > banda_superior:
    nivel_alerta = '🔴 ALERTA ALTA — TC por encima de banda superior proyectada'
elif tc_actual < banda_inferior:
    nivel_alerta = '🔴 ALERTA ALTA — TC por debajo de banda inferior proyectada'
elif vol_7d > 2 * vol_historica:
    nivel_alerta = '🟡 ALERTA MEDIA — Volatilidad reciente elevada'
else:
    nivel_alerta = '🟢 NORMAL — TC dentro de bandas esperadas'

print(f'Estado actual: {nivel_alerta}')
print(f'  TC actual:        {tc_actual:.4f}')
print(f'  Banda superior:   {banda_superior:.4f}')
print(f'  Banda inferior:   {banda_inferior:.4f}')
print(f'  Vol. 7d / hist:   {vol_7d/vol_historica:.2f}x')

In [ ]:
# ─── GRÁFICA FINAL CON FORECAST Y ALERTAS ────────────────────────────────────
# Mostrar últimos 60 días + forecast 14 días
serie_reciente = serie.dropna().tail(60)

fig, ax = plt.subplots(figsize=(14, 7))

# Serie histórica reciente
ax.plot(serie_reciente.index, serie_reciente.values,
         color=COLOR_TC, lw=2.5, label='TC Paralelo (histórico)')

# Valores ajustados (in-sample)
fitted = modelo_final.fittedvalues.tail(60)
ax.plot(fitted.index, fitted.values, color='#7F8C8D', lw=1.5,
         linestyle='--', alpha=0.7, label='Ajuste in-sample')

# Línea vertical separando histórico de forecast
ax.axvline(serie.dropna().index[-1], color='gray',
            linestyle=':', lw=2, label='Hoy')

# Intervalos de confianza del forecast
ax.fill_between(fc_ci95.index, fc_ci95.iloc[:, 0], fc_ci95.iloc[:, 1],
                 alpha=0.10, color=COLOR_PRED, label='IC 95%')
ax.fill_between(fc_ci80.index, fc_ci80.iloc[:, 0], fc_ci80.iloc[:, 1],
                 alpha=0.20, color=COLOR_PRED, label='IC 80% (zona de alerta)')

# Forecast central
ax.plot(fc_mean.index, fc_mean.values, color=COLOR_PRED, lw=2.5,
         marker='D', ms=5, label=f'Forecast {HORIZONTE}d (ARIMA{mejor_orden})')

# Punto TC actual con nivel de alerta
color_punto = '#27AE60' if 'NORMAL' in nivel_alerta else \
              ('#F39C12' if 'MEDIA' in nivel_alerta else '#E74C3C')
ax.scatter([serie.dropna().index[-1]], [tc_actual],
            color=color_punto, s=200, zorder=5, label=nivel_alerta)

ax.set_title(f'Dólar Paralelo Bolivia — Forecast {HORIZONTE} días hábiles\n'
              f'Modelo ARIMA{mejor_orden} | Datos sintéticos (originales anonimizados)',
              fontweight='bold')
ax.set_ylabel('TC Paralelo (BOB/USD)')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('img/04_forecast_alertas.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla de forecast
print('\n📅 Forecast punto central (próximas 2 semanas):')
for fecha, val, lo80, hi80 in zip(
    fc_mean.index, fc_mean.values,
    fc_ci80.iloc[:, 0].values, fc_ci80.iloc[:, 1].values
):
    print(f'  {fecha.date()}  {val:.4f}  [IC80%: {lo80:.4f} — {hi80:.4f}]')

In [ ]:
# ─── RESUMEN FINAL ────────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN — SISTEMA DE ALERTAS DÓLAR PARALELO BOLIVIA')
print('=' * 60)
print(f'\nDatos: {len(serie.dropna())} observaciones')
print(f'Período: {serie.dropna().index[0].date()} → {serie.dropna().index[-1].date()}')
print(f'\nModelo seleccionado: ARIMA{mejor_orden} (menor AIC)')
print(f'Ljung-Box p-value: {lb_p:.4f} ({"sin autocorrelación residual ✅" if lb_p > 0.05 else "revisar modelo ⚠️"})')
print(f'\nEstado actual: {nivel_alerta}')
print(f'Forecast próximos 14 días:')
print(f'  P50 (central): {fc_mean.values[0]:.4f} → {fc_mean.values[-1]:.4f}')
print(f'  IC 80% día 1:  [{fc_ci80.iloc[0,0]:.4f}, {fc_ci80.iloc[0,1]:.4f}]')
print(f'  IC 80% día 14: [{fc_ci80.iloc[-1,0]:.4f}, {fc_ci80.iloc[-1,1]:.4f}]')
print(f'\n💡 Por qué ARIMA y no LSTM:')
print(f'  Con ~{len(serie.dropna())} observaciones, LSTM necesitaría más datos para generalizar.')
print(f'  ARIMA es interpretable, rápido de reentrenar y suficientemente preciso')
print(f'  para el horizonte de 14 días requerido por Kapitalya.')